# Qwen 2.5 3B — Compound TOE-to-CPE Extraction (Colab / Kaggle Quickstart)

Run this notebook on a **free Google Colab T4 GPU (15 GB VRAM)** or **Kaggle T4/P100 GPU (16 GB VRAM)** — no local GPU required.

> **Before running:** In Colab, go to **Runtime → Change runtime type → T4 GPU**.

In [ ]:
# 1. Clone the GitHub repository and install dependencies
REPO_URL = "https://github.com/yasirdemircan/peft-cpe-extraction-pipeline.git"

import os
if not os.path.exists("repo"):
    !git clone {REPO_URL} repo
%cd repo
# Remove Colab's outdated preinstalled torchao (conflicts with latest peft)
!pip uninstall -y -q torchao
!pip install -q -r requirements.txt gdown

import torch
print("CUDA available:", torch.cuda.is_available(), "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# 2. Download the pre-trained baseline LoRA adapter from Google Drive
import os, zipfile

LORA_GDRIVE_ID = "1eKyqTdDT36zPEpi6y62kZg9V8vTmJVaX"
LORA_DIR = "checkpoints/qwen25_3b_cpe_lora"
os.makedirs(LORA_DIR, exist_ok=True)

!gdown --id {LORA_GDRIVE_ID} -O lora_adapter.zip

with zipfile.ZipFile("lora_adapter.zip", "r") as z:
    z.extractall(LORA_DIR)

# Handle case where zip contains a single nested folder
entries = os.listdir(LORA_DIR)
if len(entries) == 1 and os.path.isdir(os.path.join(LORA_DIR, entries[0])):
    nested = os.path.join(LORA_DIR, entries[0])
    for f in os.listdir(nested):
        os.rename(os.path.join(nested, f), os.path.join(LORA_DIR, f))
    os.rmdir(nested)

print("LoRA adapter files:", os.listdir(LORA_DIR))

In [ ]:
# 3. Load Base Qwen2.5-3B + Switchable LoRA Adapter (auto-downloads base model from HF)
import sys
sys.path.insert(0, "src")

from cpe_resolver import CPEDictionaryResolver
from interactive_cpe_shell import load_switchable_model, run_single_extraction, print_result

resolver = CPEDictionaryResolver()
hf_model, outlines_model, has_lora = load_switchable_model(
    "Qwen/Qwen2.5-3B-Instruct",
    "checkpoints/qwen25_3b_cpe_lora",
)

In [ ]:
# 4. Compare Baseline (Zero-Shot) vs. Fine-Tuned LoRA on any arbitrary TOE string!
test_text = "Fuji Xerox DocuCentre-IV C2265/C2263 Series Controller Software for Asia Pacific Version: Controller ROM Ver. 1.101.12"
attach_fw_to_models = False  # Set True for model-bound `_firmware` CPEs

b_parsed, b_cpes, b_time = run_single_extraction(
    hf_model, outlines_model, resolver, test_text,
    use_lora=False, has_lora=has_lora, attach_fw=attach_fw_to_models
)
print_result("BASELINE (Zero-Shot Qwen2.5-3B)", test_text, b_parsed, b_cpes, b_time, attach_fw=attach_fw_to_models)

ft_parsed, ft_cpes, ft_time = run_single_extraction(
    hf_model, outlines_model, resolver, test_text,
    use_lora=True, has_lora=has_lora, attach_fw=attach_fw_to_models
)
print_result("FINE-TUNED (QLoRA)", test_text, ft_parsed, ft_cpes, ft_time, attach_fw=attach_fw_to_models)

In [ ]:
# 5. (Optional) Run batch evaluation on the benchmark files
!python src/infer_and_resolve.py \
  --lora_dir checkpoints/qwen25_3b_cpe_lora \
  --input_file data/benchmarks/names.json \
  --limit 10 \
  --output_file outputs/names_predictions.json

In [ ]:
# 6. (Optional) Train your own improved QLoRA adapter on T4/P100 GPU
# Note: Restart runtime first if you already loaded the inference model above to free VRAM.
# !python src/train_qwen.py \
#   --train_file data/synthetic/synthetic_train.jsonl \
#   --val_file data/synthetic/synthetic_val.jsonl \
#   --output_dir checkpoints/my_improved_lora \
#   --epochs 1 --batch_size 2 --grad_accum 8